# Processamento de Linguagem Natural — Aulas 01 a 06

Notebook único: as células seguem a ordem das aulas e compartilham variáveis entre elas. Execute de cima para baixo.

> **Colab:** a primeira célula instala o que falta. Depois de instalar, se o Colab pedir, reinicie a sessão e continue.

In [ ]:
# Instalação (só roda no Google Colab; localmente use o requirements.txt)
import sys
if "google.colab" in sys.modules:
    %pip install -q gensim pyLDAvis bertopic seqeval sentence-transformers spacy nltk
    !python -m spacy download pt_core_news_lg
import nltk
for _p in ("punkt", "punkt_tab", "stopwords", "rslp"):
    nltk.download(_p, quiet=True)

## Dados de exemplo (adicionado no notebook)

Os slides assumem um corpus já preparado (`reviews`, `reviews_corpus`, `labels`, `tokens_limpos`) que não aparece neles. Aqui geramos reviews sintéticas **uma única vez**, usadas em todas as aulas. Para usar o seu corpus real, troque estas variáveis.

In [ ]:
import re
# [notebook] Dados sintéticos de exemplo — os slides assumem um corpus "já preparado" (reviews + labels)
import random
random.seed(42)
_pos = ["Adorei o produto", "Excelente qualidade", "Entrega rápida e bem embalado", "Recomendo muito",
        "Funciona perfeitamente", "Ótimo atendimento", "Superou minhas expectativas", "Produto maravilhoso"]
_neg = ["Péssimo produto", "Chegou quebrado", "Entrega atrasada", "Não recomendo",
        "Veio com defeito", "Atendimento horrível", "Qualidade ruim", "Muito decepcionado"]
_neu = ["amei esse item", "ganhei com amor", "comprei na semana passada", "o preço estava bom", "a embalagem era simples", "usei por alguns dias",
        "chegou em casa ontem", "a cor é como na foto"]
reviews_corpus, labels = [], []
for _ in range(400):
    y_ = random.randint(0, 1)
    frases = random.sample(_pos if y_ else _neg, 2) + random.sample(_neu, 1)
    random.shuffle(frases)
    reviews_corpus.append(". ".join(frases) + ".")
    labels.append(y_)
reviews = reviews_corpus  # nome usado na aula 05
print(len(reviews_corpus), "reviews | positivas:", sum(labels))

# tokens já limpos (resultado esperado da Aula 2): minúsculas, sem pontuação nem stop words curtas
_stop = {"o", "a", "e", "de", "na", "no", "um", "uma", "do", "da", "em", "como", "meu", "minha"}
tokens_limpos = [[t for t in re.findall(r"\w+", r.lower()) if t not in _stop] for r in reviews]

---
# Aula 01

## Slide 30 — POLISSEMIA EM CÓDIGO

*Bloco 01*

In [ ]:
# A palavra "banco" tem múltiplos sentidos em português
from sklearn.feature_extraction.text import CountVectorizer
corpus = [
    "O banco aprovou meu empréstimo ontem",       # financeiro
    "Sentei no banco da praça para descansar",     # móvel
    "O banco de dados PostgreSQL caiu novamente",  # tech
]
vec = CountVectorizer()
X = vec.fit_transform(corpus)
print('Vocabulário:', vec.get_feature_names_out())
# Para o BoW, 'banco' aparece igual nos 3 documentos.
# Não há nada que distinga sentido financeiro, móvel ou tech!
print('Matriz documento-termo:')
print(X.toarray())
# A coluna de 'banco' tem 1 em todas as linhas → polissemia perdida

## Slide 49 — LEI DE ZIPF A DISTRIBUIÇÃO ESTATÍSTICA DOS TOKENS EM QUALQUER CORPUS

*Bloco 02*

In [ ]:
# Lei de Zipf na prática (em português)
# Token mais frequente em corpus de português:
# 'de' (~5% de TODOS os tokens)
# 'a' (~3%)
# 'o' (~3%)
# 'que' (~2%)
# ...
# Palavras como 'arqueólogo': aparecem 1-2 vezes
# 50% do vocabulário aparece APENAS UMA VEZ (hapax legomena)
# Implicação prática:
# Top 100 palavras = ~50% de todos os tokens do corpus
# Top 1000 palavras = ~80%
# Resto do vocabulário (milhares de palavras) = ~20%

## Slide 52 — SETUP DO AMBIENTE BIBLIOTECAS PARA COMEÇAR

*Bloco 03*

In [ ]:
# Instalar bibliotecas
# pip install nltk spacy scikit-learn
# python -m spacy download pt_core_news_lg
import nltk
import spacy
import pandas as pd
from collections import Counter
# Recursos do NLTK em português
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('rslp')          # stemmer português
# Carregar modelo do spaCy em português
nlp = spacy.load('pt_core_news_lg')
# Corpus exemplo: reviews fictícias de produtos
corpus = [
    "O produto é excelente, recomendo muito! Chegou rápido.",
    "Péssimo, o aparelho veio com defeito. Não comprem.",
    "Ótima qualidade pelo preço. Entrega rápida e bem embalado.",
    "Adorei! Funciona perfeitamente, superou minhas expectativas.",
    "Esperei 3 horas no telefone. Adorei o atendimento.",  # ironia
]

## Slide 53 — PASSO 1 — CARREGANDO O CORPUS COM NLTK TOKENIZAÇÃO E CONTAGEM

*Bloco 04*

In [ ]:
from nltk.tokenize import word_tokenize
# Tokenizar todos os documentos
todos_tokens = []
for doc in corpus:
    tokens = word_tokenize(doc.lower(), language='portuguese')
    todos_tokens.extend(tokens)
print(f'Total de tokens: {len(todos_tokens)}')
print(f'Vocabulário (únicos): {len(set(todos_tokens))}')
# Top 10 palavras mais frequentes (Lei de Zipf)
contador = Counter(todos_tokens)
print('\nTop 10 mais frequentes:')
for palavra, freq in contador.most_common(10):
    print(f'  {palavra:15s} → {freq}')
# Saída esperada:
# o, ., e, , o, com, a (stop words dominam)

## Slide 54 — PASSO 2 — ANÁLISE LINGUÍSTICA COM SPACY O PADRÃO DA INDÚSTRIA

*Bloco 05*

In [ ]:
# spaCy oferece tokenização robusta + POS tagging + lemmatization + NER
doc = nlp("O João comprou um livro na Amazon ontem.")
print(f"{'TOKEN':12s} {'LEMA':12s} {'POS':10s} {'DEP':10s}")
print('-' * 50)
for token in doc:
    print(f"{token.text:12s} {token.lemma_:12s} "
          f"{token.pos_:10s} {token.dep_:10s}")
# Saída esperada:
# TOKEN        LEMA         POS        DEP
# O            o            DET        det
# João         João         PROPN      nsubj
# comprou      comprar      VERB       ROOT
# um           um           DET        det
# livro        livro        NOUN       obj
# na           em o         ADP        case
# Amazon       Amazon       PROPN      obl
# ontem        ontem        ADV        advmod
# .            .            PUNCT      punct

## Slide 55 — SPACY: NAMED ENTITY RECOGNITION IDENTIFICANDO PESSOAS, LUGARES E ORGANIZAÇÕES

*Bloco 06*

In [ ]:
# Identificação de entidades nomeadas em português
texto = ("Leonardo da Vinci nasceu em Vinci, Itália, em 1452. "
        "Foi pintor, cientista e trabalhou para Lorenzo de Médici. "
        "Morreu em Amboise, na França.")
doc = nlp(texto)
print('Entidades encontradas:')
for ent in doc.ents:
    print(f'  {ent.text:25s} → {ent.label_}')
# Saída esperada:
# Leonardo da Vinci         → PER  (pessoa)
# Vinci                     → LOC  (local)
# Itália                    → LOC
# 1452                      → MISC
# Lorenzo de Médici         → PER
# Amboise                   → LOC
# França                    → LOC

## Slide 56 — PASSO 3 — PRIMEIRA VETORIZAÇÃO DO TEXTO PRÉ-PROCESSADO A VETORES NUMÉRICOS

*Bloco 07*

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
stopwords_pt = stopwords.words('portuguese')
# Bag of Words com remoção de stop words
vec = CountVectorizer(stop_words=stopwords_pt)
X = vec.fit_transform(corpus)
print(f'Shape da matriz: {X.shape}')  # (5 docs, N features)
print(f'Vocabulário ({len(vec.vocabulary_)} termos):')
print(vec.get_feature_names_out()[:15])
# Visualizar matriz documento-termo (esparsa!)
import pandas as pd
df_dt = pd.DataFrame(
    X.toarray(),
    columns=vec.get_feature_names_out()
)
print('\nMatriz documento-termo (primeiras colunas):')
print(df_dt.iloc[:, :8])

---
# Aula 02

## Slide 13 — REGEX EM PYTHON

*Bloco 01*

In [ ]:
import re
# Padrões básicos úteis em pré-processamento de texto
texto = "<p>Adorei!! https://link.com vc tem que ver isso :)</p>"
# Remover tags HTML
limpo = re.sub(r'<[^>]+>', ' ', texto)
# 'Adorei!! https://link.com vc tem que ver isso :)'
# Remover URLs
limpo = re.sub(r'http\S+|www\.\S+', ' ', limpo)
# 'Adorei!!  vc tem que ver isso :)'
# Remover e-mails
limpo = re.sub(r'\S+@\S+', ' ', limpo)
# Manter apenas letras (incluindo acentuadas) e espaços
limpo = re.sub(r'[^a-záàâãäéèêëíìîïóòôõöúùûüç\s]', ' ', limpo)
# 'Adorei   vc tem que ver isso  '
# Colapsar espaços múltiplos
limpo = re.sub(r'\s+', ' ', limpo).strip()
# 'Adorei vc tem que ver isso'

## Slide 14 — SETUP DO AMBIENTE

*Bloco 02*

In [ ]:
import re
import unicodedata
import nltk
import spacy
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
# Recursos NLTK em português
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('rslp', quiet=True)
# Modelo spaCy em português
# python -m spacy download pt_core_news_lg
nlp = spacy.load('pt_core_news_lg')
# Corpus exemplo: reviews sujas reais
reviews = [
    "<p>ADOREI o produto!! vc precisa cmprar tbm 😍</p>",
    "Pessimo!! veio quebrado >:( https://reclame.com",
    "Mt bom mesmo, entrega rapida! 5 estrelas pra esse vendedor.",
    "Não gostei :( voltei o item, atendimento foi pessimo tbm.",
]

## Slide 15 — LIMPEZA: HTML, URLS E CONTRAÇÕES

*Bloco 03*

In [ ]:
# Mapa de contrações comuns em PT-BR informal
CONTRACTIONS = {
    'vc': 'você', 'tb': 'também', 'tbm': 'também',
    'pq': 'por que', 'obg': 'obrigado', 'blz': 'beleza',
    'td': 'tudo', 'msm': 'mesmo', 'vlw': 'valeu',
    'n': 'não', 'naum': 'não', 'cmprar': 'comprar',
    'mt': 'muito', 'pra': 'para'
}
def limpar_texto(texto):
    texto = texto.lower()                                  # 1. Lowercase
    texto = re.sub(r'<[^>]+>', ' ', texto)                 # 2. HTML
    texto = re.sub(r'http\S+|www\.\S+', ' ', texto)        # 3. URLs
    texto = re.sub(r'\S+@\S+', ' ', texto)                 # 4. E-mails
    # 5. Expande contrações
    palavras = texto.split()
    palavras = [CONTRACTIONS.get(p, p) for p in palavras]
    return ' '.join(palavras)
print(limpar_texto(reviews[0]))
# Saída: 'adorei o produto!! você precisa comprar também 😍'

## Slide 18 — TRATAMENTO DE ACENTOS DECISÃO SENSÍVEL AO DOWNSTREAM

*Bloco 04*

In [ ]:
import unicodedata
def remove_acentos(texto):
    """Remove acentos: 'maravilhoso' → 'maravilhoso'.
    Útil para busca; perigoso para NER."""
    nfkd = unicodedata.normalize('NFKD', texto)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))
# Exemplos
print(remove_acentos('São Paulo'))      # 'Sao Paulo'
print(remove_acentos('experiência'))    # 'experiencia'
print(remove_acentos('coração'))        # 'coracao'

## Slide 20 — PIPELINE COMPLETO: LIMPAR_NORMALIZAR() A FUNÇÃO QUE COSTURA TUDO

*Bloco 05*

In [ ]:
def limpar_normalizar(texto, remove_acentos=False):
    """Pipeline modular para texto em PT-BR."""
    if not isinstance(texto, str):
        return ''
    texto = texto.lower()                                       # 1
    texto = re.sub(r'<[^>]+>', ' ', texto)                      # 2
    texto = re.sub(r'http\S+|www\.\S+', ' ', texto)             # 3
    texto = re.sub(r'\S+@\S+', ' ', texto)                      # 4
    palavras = [CONTRACTIONS.get(p, p) for p in texto.split()]  # 5
    texto = ' '.join(palavras)
    texto = re.sub(r'[^a-záàâãäéèêëíìîïóòôõöúùûüç\s.,!?]', ' ', texto)  # 6
    if remove_acentos:                                          # 7
        nfkd = unicodedata.normalize('NFKD', texto)
        texto = ''.join(c for c in nfkd if not unicodedata.combining(c))
    texto = re.sub(r'\s+', ' ', texto).strip()                  # 8
    return texto
# Teste
sujo = '<p>Adorei o produto!! https://exemplo.com VC PRECISA TBM!!!</p>'
print(limpar_normalizar(sujo))
# 'adorei o produto !! você precisa também !!!'

## Slide 30 — FAMÍLIA 1: SPLIT() INGÊNUO O BASELINE DIDÁTICO

*Bloco 06*

In [ ]:
# split() simples — divide só por espaços
texto = "O Sr. Silva, hoje, tem R$ 1.250,99 no banco!"
tokens = texto.split()
print(tokens)
# ['O', 'Sr.', 'Silva,', 'hoje,', 'tem', 'R$', '1.250,99', 'no', 'banco!']
# Problemas:
# - 'Silva,' (com vírgula) ≠ 'Silva' → tokens DIFERENTES no vocabulário
# - 'banco!' (com !) ≠ 'banco' → tokens DIFERENTES
# - 'hoje,' não separa a vírgula
# Útil só como BASELINE didático.
# Em produção: NUNCA use split() puro.

## Slide 31 — FAMÍLIA 2A: NLTK WORD_TOKENIZE ROBUSTO, COM SUPORTE A PORTUGUÊS

*Bloco 07*

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize
texto = "O Sr. Silva, hoje, tem R$ 1.250,99 no banco! Adorei."
# Tokenização de palavras (com language='portuguese')
tokens = word_tokenize(texto, language='portuguese')
print(tokens)
# ['O', 'Sr.', 'Silva', ',', 'hoje', ',', 'tem', 'R$',
#  '1.250,99', 'no', 'banco', '!', 'Adorei', '.']
# ✅ 'Sr.' reconhecido como abreviação
# ✅ Vírgulas separadas como tokens
# ✅ Número 1.250,99 mantido inteiro
# Tokenização de FRASES
frases = sent_tokenize(texto, language='portuguese')
print(frases)
# ['O Sr. Silva, hoje, tem R$ 1.250,99 no banco!', 'Adorei.']

## Slide 32 — FAMÍLIA 2B: SPACY O PADRÃO DA INDÚSTRIA MODERNA

*Bloco 08*

In [ ]:
import spacy
nlp = spacy.load('pt_core_news_lg')
texto = "O Sr. Silva, hoje, tem R$ 1.250,99 no banco!"
# spaCy: tokenização + POS + lema + dependência tudo de uma vez
doc = nlp(texto)
for tok in doc:
    print(f"{tok.text:<12} {tok.lemma_:<12} {tok.pos_}")
# Saída:
# O           o           DET
# Sr.         Sr.         PROPN
# Silva       Silva       PROPN
# ,           ,           PUNCT
# hoje        hoje        ADV
# ...
# 1.250,99    1.250,99    NUM
# no          em o        ADP
# banco       banco       NOUN
# !           !           PUNCT
# 🚀 Vantagem: tokens já vêm com lema, POS, dependência!

## Slide 33 — SENTENCE TOKENIZATION QUEBRANDO TEXTO EM FRASES

*Bloco 09*

In [ ]:
# Tokenização de FRASES (não palavras)
from nltk.tokenize import sent_tokenize
texto = ("Adorei o produto. Chegou rápido! "
        "Recomendo a todos. Mas o preço foi alto...")
frases = sent_tokenize(texto, language='portuguese')
for i, f in enumerate(frases, 1):
    print(f'{i}. {f}')
# 1. Adorei o produto.
# 2. Chegou rápido!
# 3. Recomendo a todos.
# 4. Mas o preço foi alto...
# Em spaCy é automático
for sent in nlp(texto).sents:
    print(sent.text)

## Slide 37 — BPE: BYTE-PAIR ENCODING O ALGORITMO DOS TRANSFORMERS

*Bloco 10*

In [ ]:
# Exemplo simplificado de BPE em ação
# Vocabulário inicial: caracteres individuais
# 'l', 'o', 'w', 'e', 'r', ...
# Iteração 1: 'l' + 'o' aparece muito → mescla → 'lo'
# Iteração 2: 'lo' + 'w' aparece muito → 'low'
# Iteração 3: 'e' + 'r' aparece muito → 'er'
# Após várias iterações: vocabulário tem subwords
# como 'low', 'er', 'est', 'ing', '##ing'
# Resultado para 'lowest':
# ['low', 'est']
# Para palavra rara 'newer':
# ['n', 'ewer'] ou ['new', '##er']

## Slide 40 — CURIOSIDADE POR QUE GPT E BERT TOKENIZAM DE FORMA ESTRANHA

*Bloco 11*

In [ ]:
# Tokenizer do BERTimbau (BERT em PT-BR)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')
print(tok.tokenize('correríamos pela praia'))
# ['corre', '##ría', '##mos', 'pela', 'praia']
print(tok.tokenize('inconstitucionalissimamente'))
# ['in', '##cons', '##ti', '##tu', '##cion', '##ali', '##ss', '##im', '##amente']

## Slide 50 — RSLP EM AÇÃO O STEMMER BRASILEIRO

*Bloco 12*

In [ ]:
from nltk.stem import RSLPStemmer
import nltk
nltk.download('rslp', quiet=True)
stemmer = RSLPStemmer()
exemplos = ['correndo', 'correu', 'correriam',
            'maravilhoso', 'felicidades', 'menino',
            'meninas', 'casinha', 'gatos']
for palavra in exemplos:
    print(f'{palavra:<15} → {stemmer.stem(palavra)}')
# Saída:
# correndo        → corr
# correu          → corr
# correriam       → corr
# maravilhoso     → maravilh
# felicidades     → felic
# menino          → menin
# meninas         → menin
# casinha         → casinh
# gatos           → gat

## Slide 51 — LEMMATIZATION COM SPACY MAIS LENTO, MAIS PRECISO

*Bloco 13*

In [ ]:
import spacy
nlp = spacy.load('pt_core_news_lg')
texto = 'As crianças estavam correndo felizes pela praia maravilhosa.'
doc = nlp(texto)
for token in doc:
    print(f'{token.text:<14} → {token.lemma_}')
# Saída:
# As             → o
# crianças       → criança
# estavam        → estar
# correndo       → correr
# felizes        → feliz
# pela           → por o
# praia          → praia
# maravilhosa    → maravilhoso
# .              → .
# 🎯 Repare: lemmatization PRESERVA palavras válidas
# 'correr' (não 'corr'), 'maravilhoso' (não 'maravilh')

## Slide 54 — O QUE SÃO STOP WORDS? PALAVRAS FREQUENTES COM POUCO CONTEÚDO SEMÂNTICO

*Bloco 14*

In [ ]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)
STOP_WORDS_PT = set(stopwords.words('portuguese'))
print(f'Total: {len(STOP_WORDS_PT)} stop words')
# Total: 207 stop words
print('Algumas:', list(STOP_WORDS_PT)[:15])
# ['o', 'a', 'os', 'as', 'de', 'do', 'da', 'em', 'no', 'na',
#  'que', 'um', 'uma', 'para', 'com', ...]

## Slide 57 — PIPELINE NLP FINAL TUDO COSTURADO EM UMA FUNÇÃO

*Bloco 15*

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer
STOP_WORDS_PT = set(stopwords.words('portuguese'))
stemmer = RSLPStemmer()
def pipeline_nlp(textos: list[str]) -> list[list[str]]:
    """Lista de textos PT → lista de listas de tokens limpos."""
    resultado = []
    for texto in textos:
        texto = limpar_normalizar(texto)                              # 1
        tokens = word_tokenize(texto, language='portuguese')          # 2
        tokens = [t for t in tokens                                   # 3
                  if t not in STOP_WORDS_PT and len(t) > 2]
        tokens = [stemmer.stem(t) for t in tokens]                    # 4
        resultado.append(tokens)
    return resultado
reviews = ['Adorei o produto, entrega rapidíssima!',
           'Não gostei, veio quebrado e atrasado.']
print(pipeline_nlp(reviews))
# [['ador', 'produt', 'entreg', 'rap'],
#  ['gost', 'vei', 'quebr', 'atras']]

---
# Aula 03

## Slide 15 — BOW MANUALMENTE EM PYTHON SEM LIBS, PARA ENTENDER POR DENTRO

*Bloco 01*

In [ ]:
# BoW manual — 30 linhas para entender o que sklearn faz
from collections import Counter
corpus = [
    'Adorei o produto, entrega rápida',
    'Produto chegou quebrado',
    'Entrega rápida, atendimento bom',
]
# 1. Tokenizar (simplificado)
docs_tokens = [doc.lower().replace(',','').split() for doc in corpus]
# 2. Construir vocabulário (palavras únicas ordenadas)
vocab = sorted(set(t for doc in docs_tokens for t in doc))
print(f'Vocabulário ({len(vocab)} palavras):', vocab)
# 3. Construir matriz documento-termo
matriz = []
for tokens in docs_tokens:
    contagem = Counter(tokens)
    linha = [contagem.get(p, 0) for p in vocab]
    matriz.append(linha)
import pandas as pd
df = pd.DataFrame(matriz, columns=vocab)
print(df)

## Slide 16 — BOW COM SCIKIT-LEARN COUNTVECTORIZER FAZ TUDO

*Bloco 02*

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
corpus = [
    'Adorei o produto, entrega rápida e qualidade boa.',
    'Produto chegou quebrado, qualidade péssima.',
    'Entrega rápida, atendimento bom, recomendo!',
    'Produto bom mas atendimento péssimo.',
]
# CountVectorizer: tokenização + vocabulário + matriz
vectorizer = CountVectorizer(lowercase=True, max_features=20)
X = vectorizer.fit_transform(corpus)
vocab = vectorizer.get_feature_names_out()
df = pd.DataFrame(X.toarray(), columns=vocab)
print(df)
print(f'Tamanho do vocabulário: {len(vocab)}')
print(f'Esparsidade: {1 - X.nnz / (X.shape[0] * X.shape[1]):.2%}')

## Slide 17 — SAÍDA TÍPICA DO COUNTVECTORIZER MATRIZ DOCUMENTO-TERMO EM AÇÃO

*Bloco 03*

In [ ]:
# Saída ilustrativa:
#    adorei  atendimento  bom  boa  chegou  entrega  mas  o  ...
# 0       1            0    0    1       0        1    0  1  ...
# 1       0            0    0    0       1        0    0  0  ...
# 2       0            1    1    0       0        1    0  0  ...
# 3       0            1    1    0       0        0    1  0  ...
# Tamanho do vocabulário: 17
# Esparsidade: 73.53%
# Cada linha: um documento
# Cada coluna: uma palavra do vocabulário
# Valores: contagem da palavra no documento
# Cuidado: lowercase=True é o default
# Outros parâmetros úteis:
#   - stop_words: lista para remover
#   - max_features: limita N palavras mais frequentes
#   - min_df / max_df: filtros por frequência

## Slide 18 — ESPARSIDADE MATRIZ CHEIA DE ZEROS

*Bloco 04*

In [ ]:
# Visualizando a esparsidade
print(f'Shape: {X.shape}')                # (10000, 30000)
print(f'Total de células: {X.shape[0] * X.shape[1]:,}')  # 300,000,000
print(f'Não-zeros: {X.nnz:,}')             # ~1,500,000
print(f'Esparsidade: {1 - X.nnz/(X.shape[0]*X.shape[1]):.2%}')
# Esparsidade: 99.50%
# Armazenar matriz cheia em numpy: ~2.4 GB (300M × 8 bytes)
# Armazenar como esparsa: ~36 MB (1.5M × 24 bytes por trio)

## Slide 19 — SCIPY.SPARSE

*Bloco 05*

In [ ]:
# [notebook] rótulos do corpus de 4 reviews do slide 16 (1=pos, 0=neg)
y = [1, 0, 1, 0]
from scipy.sparse import csr_matrix
import numpy as np
# X já vem como matriz esparsa do CountVectorizer
print(type(X))  # scipy.sparse.csr_matrix
# Usar com modelos de sklearn — todos suportam esparsa nativamente
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X, y)  # ✅ Funciona direto, sem converter
# Converter para denso APENAS se precisar visualizar
# (cuidado: pode estourar memória em corpora grandes!)
X_dense = X.toarray()  # ⚠️ Use só com X pequeno

## Slide 31 — TRIGRAMS (N=3)

*Bloco 06*

In [ ]:
# Trigrams capturam expressões idiomáticas e contexto mais longo
from sklearn.feature_extraction.text import CountVectorizer
textos = [
    'não gostei nada do produto',
    'muito bom mesmo, recomendo',
    'foi de jeito nenhum',
]
# ngram_range=(3, 3) → APENAS trigrams
vec = CountVectorizer(ngram_range=(3, 3))
X = vec.fit_transform(textos)
print(vec.get_feature_names_out())
# Saída:
# ['de jeito nenhum', 'do produto'... 'mesmo recomendo',
#  'muito bom mesmo', 'não gostei nada']
# Trigrams capturam:
# - Expressões fixas: 'de jeito nenhum'
# - Modificadores intensivos: 'muito bom mesmo'
# - Negações compostas: 'não gostei nada'

## Slide 32 — RANGE (1, 2): UNIGRAMS + BIGRAMS

*Bloco 07*

In [ ]:
# CountVectorizer com unigrams + bigrams
vec = CountVectorizer(ngram_range=(1, 2), min_df=2)
X = vec.fit_transform(corpus)
# Vocabulário agora inclui AMBOS:
#   ['adorei', 'adorei produto', 'bom', 'entrega',
#    'entrega rápida', 'gostei', 'não', 'não gostei',
#    'produto', 'rápida', ...]
# Em sentiment analysis em PT-BR:
# - Unigrams capturam 'gostei', 'péssimo', 'maravilhoso'
# - Bigrams capturam 'não gostei', 'muito bom', 'super ruim'
print(f'Total de features: {len(vec.get_feature_names_out())}')

## Slide 33 — IMPLEMENTAÇÃO PRÁTICA

*Bloco 08*

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
corpus = [
    'Adorei o produto, super recomendo',
    'Não gostei muito do produto',
    'Produto bom mesmo, vale o preço',
    'Muito ruim, não comprem',
]
# Comparando ngram_range diferentes
for n_range in [(1, 1), (1, 2), (1, 3), (2, 2)]:
    vec = CountVectorizer(ngram_range=n_range)
    X = vec.fit_transform(corpus)
    print(f'ngram_range={n_range}: {X.shape[1]} features')
# Saída típica:
# ngram_range=(1, 1): 18 features    ← unigrams só
# ngram_range=(1, 2): 35 features    ← +17 bigrams
# ngram_range=(1, 3): 50 features    ← +15 trigrams
# ngram_range=(2, 2): 17 features    ← APENAS bigrams

## Slide 35 — FILTROS: MIN_DF E MAX_DF

*Bloco 09*

In [ ]:
# Filtros para podar n-grams ruidosos ou inúteis
vec = CountVectorizer(
    ngram_range=(1, 2),
    min_df=5,        # ignora termos em MENOS de 5 documentos
    max_df=0.85,     # ignora termos em MAIS de 85% dos documentos
    max_features=10000,  # limita vocabulário aos 10k mais frequentes
)
X = vec.fit_transform(reviews_corpus)  # [notebook] corpus maior, p/ min_df=5 fazer sentido
# Por que filtrar?
# - min_df remove n-grams raros (typos, palavras únicas) → reduz ruído
# - max_df remove palavras universais (alternativa a stop words)
# - max_features cap absoluto no tamanho do vocabulário
# Exemplo: corpus de 10k docs
# Sem filtros: ~600k features (lento!)
# Com filtros acima: ~10k features (manejável)

## Slide 38 — BIGRAMS + SENTIMENT ANALYSIS EM PT-BR

*Bloco 10*

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
# Corpus de reviews + labels (0=neg, 1=pos)
# reviews_corpus = [...]; labels = [...]
# Sem bigrams (unigrams puros)
bow_uni = CountVectorizer(ngram_range=(1, 1), min_df=5)
X_uni = bow_uni.fit_transform(reviews_corpus)
f1_uni = cross_val_score(LogisticRegression(max_iter=1000),
                          X_uni, labels, cv=5, scoring='f1').mean()
# Com unigrams + bigrams
bow_bi = CountVectorizer(ngram_range=(1, 2), min_df=5)
X_bi = bow_bi.fit_transform(reviews_corpus)
f1_bi = cross_val_score(LogisticRegression(max_iter=1000),
                         X_bi, labels, cv=5, scoring='f1').mean()
print(f'Unigrams puros:    F1 = {f1_uni:.3f}')
print(f'Unigrams + bigrams: F1 = {f1_bi:.3f}')
print(f'Ganho: +{(f1_bi - f1_uni)*100:.1f} pontos')
# Tipicamente: ganho de 2-5 pontos em sentiment PT-BR

## Slide 48 — TF: TERM FREQUENCY

*Bloco 11*

In [ ]:
def tf(palavra, doc):
    """Term Frequency normalizado pelo tamanho do doc."""
    palavras = doc.split()
    return palavras.count(palavra) / len(palavras)
doc = 'o produto bom o produto rápido'
# 6 palavras totais
print(tf('o', doc))         # 2/6 = 0.333
print(tf('produto', doc))   # 2/6 = 0.333
print(tf('bom', doc))       # 1/6 = 0.167
print(tf('rápido', doc))    # 1/6 = 0.167

## Slide 49 — IDF: INVERSE DOCUMENT FREQUENCY

*Bloco 12*

In [ ]:
import numpy as np
def idf(palavra, corpus):
    """Inverse Document Frequency."""
    df = sum(1 for doc in corpus if palavra in doc.split())
    return np.log(len(corpus) / df) if df > 0 else 0
corpus = [
    'o produto bom',
    'o produto ruim',
    'o produto quebrado',
    'o produto rápido',
]
print(idf('o', corpus))         # log(4/4) = 0     ← em TODO doc
print(idf('produto', corpus))   # log(4/4) = 0     ← em TODO doc
print(idf('bom', corpus))       # log(4/1) = 1.39  ← só em 1 doc

## Slide 51 — TF-IDF MANUALMENTE

*Bloco 13*

In [ ]:
import numpy as np
import pandas as pd
corpus = [
    'produto bom rapida entrega',
    'produto ruim quebrado',
    'entrega rapida atendimento bom',
    'atendimento ruim produto bom',
]
vocab = sorted(set(p for doc in corpus for p in doc.split()))
def tf(p, doc):
    palavras = doc.split()
    return palavras.count(p) / len(palavras)
def idf(p, corpus):
    df = sum(1 for doc in corpus if p in doc.split())
    return np.log(len(corpus) / df) if df > 0 else 0
# Calcular matriz TF-IDF
matriz = []
for doc in corpus:
    linha = [tf(p, doc) * idf(p, corpus) for p in vocab]
    matriz.append(linha)
df_tfidf = pd.DataFrame(matriz, columns=vocab)
print(df_tfidf.round(3))

## Slide 52 — SAÍDA DO TF-IDF MANUAL

*Bloco 14*

In [ ]:
# Saída ilustrativa (arredondada):
#                atendimento    bom  entrega  produto  quebrado  rapida  ruim
# Doc 1               0.000     0.072  0.072    0.072     0.000   0.072  0.000
# Doc 2               0.000     0.000  0.000    0.096     0.462   0.000  0.231
# Doc 3               0.173     0.072  0.072    0.000     0.000   0.072  0.000
# Doc 4               0.173     0.072  0.000    0.072     0.000   0.000  0.231
# Repare como cada palavra recebe peso DIFERENTE em cada doc:
# - 'produto' aparece em 3/4 docs → IDF baixo → peso BAIXO
# - 'quebrado' aparece em 1/4 docs → IDF ALTO → peso 0.462 (TOP!)
# - 'atendimento' aparece em 2/4 docs → IDF médio
# A palavra MAIS DISCRIMINATIVA do Doc 2 é 'quebrado' — exatamente o que faz
# sentido! É a palavra que SINALIZA o problema do produto.

## Slide 53 — TFIDFVECTORIZER EM PRODUÇÃO

*Bloco 15*

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),         # unigrams + bigrams
    min_df=5,                   # ignora termos em <5 docs
    max_df=0.85,                # ignora termos em >85% dos docs
    max_features=10000,         # cap no vocabulário
    sublinear_tf=True,          # aplica log(1+TF) em vez de TF puro
)
X = vectorizer.fit_transform(reviews_corpus)  # 10k documentos
print(X.shape)        # (10000, 10000)
print(X.nnz)          # número de elementos não-zero
# Top features por documento
import numpy as np
feature_names = vectorizer.get_feature_names_out()
doc0 = X[0].toarray().flatten()
top_idx = np.argsort(doc0)[-5:][::-1]
print('Top 5 features do doc 0:')
for i in top_idx:
    print(f'  {feature_names[i]}: {doc0[i]:.3f}')

## Slide 55 — BOW VS TF-IDF: TAREFA REAL

*Bloco 16*

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
# Hipotético: reviews_corpus e labels (0=neg, 1=pos) já preparados
# 1) BoW
bow = CountVectorizer(ngram_range=(1, 2), min_df=5)
X_bow = bow.fit_transform(reviews_corpus)
score_bow = cross_val_score(
    LogisticRegression(max_iter=1000), X_bow, labels,
    cv=5, scoring='f1'
).mean()
# 2) TF-IDF
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True)
X_tfidf = tfidf.fit_transform(reviews_corpus)
score_tfidf = cross_val_score(
    LogisticRegression(max_iter=1000), X_tfidf, labels,
    cv=5, scoring='f1'
).mean()
print(f'BoW    F1: {score_bow:.3f}')   # ex: 0.812
print(f'TF-IDF F1: {score_tfidf:.3f}') # ex: 0.834 — geralmente vence

---
# Aula 04

## Slide 8 — EMBEDDINGS DENSOS POUCOS NÚMEROS, MUITA SEMÂNTICA

*Bloco 01*

In [ ]:
# Em embeddings (300 dimensões)
vec_rei      = [0.42, -0.13, 0.78, 0.21, ..., -0.05]   # 300 floats
vec_rainha   = [0.41, -0.18, 0.75, 0.19, ..., -0.07]   # 300 floats
vec_abacaxi  = [-0.31, 0.62, 0.04, -0.55, ..., 0.88]   # 300 floats
# Similaridade do cosseno:
# cos_sim(rei, rainha)  = 0.72   # ALTAMENTE similares!
# cos_sim(rei, abacaxi) = 0.05   # praticamente sem relação
# Espaço vetorial onde GEOMETRIA = SEMÂNTICA

In [ ]:
# [notebook] a Aula 02 reutilizou o nome `reviews` com listas pequenas; restauramos o corpus de exemplo
reviews = reviews_corpus

## Slide 17 — TREINANDO WORD2VEC COM GENSIM SETUP E TREINAMENTO EM PT-BR

*Bloco 02*

In [ ]:
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt', quiet=True)
# Hipotético: 'reviews' é uma lista de strings em PT
# Tokenizamos cada review (já com pré-processamento da Aula 2)
sentences = [word_tokenize(r.lower(), language='portuguese')
             for r in reviews]
# Treina Word2Vec com Skip-gram + negative sampling
model = Word2Vec(
    sentences=sentences,
    vector_size=100,        # dimensionalidade do embedding
    window=5,               # janela de contexto
    min_count=5,            # ignora palavras com freq < 5
    workers=4,              # paralelismo
    sg=1,                   # 1 = Skip-gram, 0 = CBOW
    negative=10,            # negative sampling (10 amostras)
    epochs=10,
)
model.save('word2vec_ptbr.model')

## Slide 18 — USANDO O MODELO TREINADO ACESSANDO VETORES E SIMILARIDADES

*Bloco 03*

In [ ]:
# Vetor de uma palavra
print(model.wv['produto'][:5])
# Saída: array([0.123, -0.456, 0.789, 0.234, -0.567])
# Tamanho do vetor (vector_size=100)
print(model.wv['produto'].shape)
# Saída: (100,)
# Palavras mais similares
print(model.wv.most_similar('produto', topn=5))
# Saída:
# [('item', 0.78),
#  ('aparelho', 0.74),
#  ('mercadoria', 0.69),
#  ('artigo', 0.65),
#  ('peça', 0.61)]
# Similaridade entre duas palavras
print(model.wv.similarity('produto', 'item'))     # 0.78
print(model.wv.similarity('produto', 'amor'))     # 0.12
# Carregar modelo salvo
model = Word2Vec.load('word2vec_ptbr.model')

> ⚠️ **Atenção (adicionado no notebook):** a partir daqui as células de FastText usam o modelo oficial `wiki.pt.bin` (~7 GB, ~7 GB de RAM). Baixe antes de rodar:
> `https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.pt.bin` e salve na mesma pasta do notebook. Sem o arquivo, essas células dão `FileNotFoundError`. O slide 38 (fine-tuning) é ilustrativo: precisa de `nilc-w2v-skip-300.txt` e de um `meu_corpus_dominio` seu.

## Slide 31 — FASTTEXT RESOLVE OOV NÃO HÁ MAIS "PALAVRA DESCONHECIDA"

*Bloco 04*

In [ ]:
from gensim.models.fasttext import load_facebook_vectors
# Modelo FastText oficial em português
modelo = load_facebook_vectors('wiki.pt.bin')
# Vetor de palavra comum
v = modelo['amor']
print(v.shape)  # (300,)
# Vetor de NEOLOGISMO que NUNCA esteve no corpus
v_novo = modelo['instagramável']
print(v_novo.shape)  # (300,) — foi inferido pelos subwords!
# Vetor de gíria
v_giria = modelo['baita']
print(v_giria.shape)  # (300,) — funciona
# Em Word2Vec, isso jogaria KeyError. Em FastText, sempre tem vetor.

## Slide 36 — FASTTEXT OFICIAL (FACEBOOK) 157 LÍNGUAS, INCLUINDO PT

*Bloco 05*

In [ ]:
# Download do modelo PT (~7 GB)
# wget https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.pt.bin
# Carregando com Gensim
from gensim.models.fasttext import load_facebook_vectors
modelo = load_facebook_vectors('wiki.pt.bin')
print(modelo['celular'].shape)   # (300,)
print(modelo.most_similar('celular', topn=5))
# [('smartphone', 0.81), ('telefone', 0.78),
#  ('aparelho', 0.71), ('android', 0.68), ('iphone', 0.65)]
# Versão menor (.vec) só com vetores principais ~1.5 GB
# https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.pt.vec

## Slide 38 — QUANDO VALE FINE-TUNING? ADAPTANDO PRÉ-TREINADOS AO SEU DOMÍNIO

*Bloco 06*

In [ ]:
from gensim.models import Word2Vec, KeyedVectors
# Carrega embeddings pré-treinados
base = KeyedVectors.load_word2vec_format('nilc-w2v-skip-300.txt')
# Inicia novo modelo com os pesos pré-treinados
model = Word2Vec(vector_size=300, min_count=5, sg=1)
model.build_vocab(meu_corpus_dominio)
model.wv.vectors = np.copy(base.vectors)  # transfere pesos
# Continua o treinamento no SEU corpus (fine-tuning)
model.train(meu_corpus_dominio, total_examples=len(meu_corpus_dominio),
            epochs=5)

## Slide 47 — CARREGANDO EMBEDDINGS PRÉ-TREINADOS FASTTEXT OFICIAL EM PT-BR

*Bloco 07*

In [ ]:
from gensim.models.fasttext import load_facebook_vectors
# Baixe antes:
# wget https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.pt.bin
# Carregando o modelo (pode demorar ~30s, ocupa ~7GB de RAM)
modelo = load_facebook_vectors('wiki.pt.bin')
# Vetor de uma palavra
v = modelo['amor']
print(v.shape)  # (300,)
print(v[:5])    # array([0.123, -0.456, 0.789, 0.234, -0.567])
# Funciona até com OOV graças a subwords:
v_neologismo = modelo['instagramável']
print(v_neologismo.shape)  # (300,) - foi inferido pelos subwords
# Tamanho do vocabulário
print(f'Vocabulário: {len(modelo.key_to_index):,} palavras')
# ~2 milhões de palavras conhecidas + qualquer OOV via subword

## Slide 49 — SIMILARIDADE DO COSSENO EM CÓDIGO COM GENSIM

*Bloco 08*

In [ ]:
# Similaridade entre duas palavras
print(modelo.similarity('rei', 'rainha'))      # ~0.72
print(modelo.similarity('rei', 'abacaxi'))     # ~0.05
print(modelo.similarity('feliz', 'alegre'))    # ~0.78
print(modelo.similarity('feliz', 'triste'))    # ~0.45 (relacionados, opostos!)
# Palavras mais similares (top-k)
print(modelo.most_similar('celular', topn=5))
# [('smartphone', 0.81),
#  ('telefone', 0.78),
#  ('aparelho', 0.71),
#  ('android', 0.68),
#  ('iphone', 0.65)]
# Em PT-BR
print(modelo.most_similar('feliz', topn=5))
# [('alegre', 0.78), ('contente', 0.74), ('eufórico', 0.71),
#  ('animado', 0.68), ('satisfeito', 0.65)]

## Slide 51 — ANALOGIAS EM CÓDIGO "REI ESTÁ PARA HOMEM COMO X ESTÁ PARA MULHER"

*Bloco 09*

In [ ]:
# A famosa analogia: rei está para homem assim como X está para mulher
# Aritmética: vec(rei) - vec(homem) + vec(mulher) = ?
resultado = modelo.most_similar(
    positive=['rei', 'mulher'],   # adiciona
    negative=['homem'],            # subtrai
    topn=3
)
print(resultado)
# [('rainha', 0.78),
#  ('princesa', 0.72),
#  ('imperatriz', 0.68)]
# Por que funciona?
# A direção homem→mulher no espaço vetorial codifica a relação masculino→feminino
# Aplicada a 'rei', leva ao equivalente feminino: 'rainha'
# Outras analogias:
modelo.most_similar(
    positive=['paris', 'alemanha'],
    negative=['frança'],
    topn=3
)
# [('berlim', 0.74), ('hamburgo', 0.62), ('munique', 0.58)]

## Slide 56 — VISUALIZAÇÃO EM CÓDIGO CLUSTERS SEMÂNTICOS COM T-SNE

*Bloco 10*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
# Selecionamos um conjunto de palavras de interesse
palavras = ['cachorro', 'gato', 'leão', 'tigre', 'elefante',
            'carro', 'moto', 'ônibus', 'caminhão', 'bicicleta',
            'feliz', 'triste', 'alegre', 'deprimido', 'eufórico']
# Cada palavra tem vetor de 300 dimensões
vetores = np.array([modelo[p] for p in palavras])
print(vetores.shape)  # (15, 300)
# Reduz para 2D com t-SNE
tsne = TSNE(n_components=2, perplexity=5, random_state=42)
vetores_2d = tsne.fit_transform(vetores)
print(vetores_2d.shape)  # (15, 2)
# Plota
plt.figure(figsize=(10, 8))
plt.scatter(vetores_2d[:, 0], vetores_2d[:, 1])
for i, palavra in enumerate(palavras):
    plt.annotate(palavra, (vetores_2d[i, 0], vetores_2d[i, 1]))
plt.title('Clusters semânticos em PT-BR (FastText)')
plt.show()

---
# Aula 05

## Slide 12 — SUAVIZAÇÃO DE LAPLACE EVITA PROBABILIDADES ZERO

*Bloco 01*

In [ ]:
# Suavização de Laplace (add-one smoothing)
# Adiciona alpha (default=1.0) ao numerador,
# e alpha * |vocab| ao denominador
# Antes da suavização:
# P('quebrado' | positivo) = 0 / 1000 = 0  ← problema
# Depois da suavização (alpha=1):
# P('quebrado' | positivo) = (0 + 1) / (1000 + |vocab|)
#                          = 1 / 31000
#                          ≈ 0.000032  ← pequeno mas não zero
from sklearn.naive_bayes import MultinomialNB
clf = MultinomialNB(alpha=1.0)  # alpha = força da suavização

In [ ]:
# [notebook] a Aula 02 reutilizou o nome `reviews` com listas pequenas; restauramos o corpus de exemplo
reviews = reviews_corpus

## Slide 14 — MULTINOMIAL NB + TF-IDF IMPLEMENTAÇÃO EM PIPELINE

*Bloco 02*

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
# Hipotético: 'reviews' é lista de strings, 'labels' é lista 0/1
X_train, X_test, y_train, y_test = train_test_split(
    reviews, labels, test_size=0.2, stratify=labels, random_state=42
)
# Pipeline = vetorização + classificador em UM ÚNICO objeto
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True)),
    ('clf', MultinomialNB(alpha=1.0)),
])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['negativo', 'positivo']))

## Slide 15 — SAÍDA DO CLASSIFICATION_REPORT MÉTRICAS POR CLASSE

*Bloco 03*

In [ ]:
# Saída ilustrativa do classification_report:
#               precision    recall  f1-score   support
#     negativo       0.85      0.82      0.83      1000
#     positivo       0.83      0.86      0.84      1000
#     accuracy                           0.84      2000
#    macro avg       0.84      0.84      0.84      2000
# weighted avg       0.84      0.84      0.84      2000
# Como ler:
# - precision: das previsões positivas, quantas certas?
# - recall: dos positivos reais, quantos encontramos?
# - f1-score: harmônica de precision e recall
# - support: número de exemplos por classe
# - macro avg: média simples (todas classes pesam igual)
# - weighted avg: ponderada pelo support

## Slide 17 — REGRESSÃO LOGÍSTICA O CAVALO-DE-BATALHA DA CLASSIFICAÇÃO

*Bloco 04*

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', LogisticRegression(
        max_iter=1000,
        solver='liblinear',     # rápido para texto esparso
        C=1.0,                  # regularização (menor C = mais regularização)
        class_weight='balanced'  # ajusta para classes desbalanceadas
    )),
])
pipe.fit(X_train, y_train)

## Slide 18 — INTERPRETABILIDADE DOS COEFICIENTES A GRANDE VANTAGEM DA LR

*Bloco 05*

In [ ]:
# Acessando coeficientes do modelo treinado
import numpy as np
vocab = pipe.named_steps['tfidf'].get_feature_names_out()
coefs = pipe.named_steps['clf'].coef_[0]
# Top 10 palavras mais POSITIVAS (coef alto)
top_pos = np.argsort(coefs)[-10:][::-1]
print('Mais associadas a POSITIVO:')
for i in top_pos:
    print(f'  {vocab[i]:<20s} ({coefs[i]:+.3f})')
# Saída:
#   adorei         (+2.450)
#   excelente      (+2.103)
#   maravilhoso    (+1.987)
#   recomendo      (+1.812)
#   perfeito       (+1.654)
# Top 10 NEGATIVAS
top_neg = np.argsort(coefs)[:10]
print('Mais associadas a NEGATIVO:')
# pessimo, ruim, quebrado, defeituoso, horrivel, ...

## Slide 29 — CLASS_WEIGHT='BALANCED' LIDANDO COM CLASSES DESBALANCEADAS

*Bloco 06*

In [ ]:
# Sem class_weight: modelo prevê tudo como majoritária
LogisticRegression()
# Recall na classe minoritária: ~5%  ← inútil
# Com class_weight='balanced': pesos inversos à frequência
LogisticRegression(class_weight='balanced')
# Recall na classe minoritária: ~75%  ← funcional!
# Equivalente:
# class_weight={0: 1.0, 1: 19.0}  # se 1 é 5% dos dados
# Outras alternativas:
# - Oversampling: SMOTE da imblearn
# - Undersampling: RandomUnderSampler
# - Modelos com built-in support: XGBoost com scale_pos_weight

## Slide 32 — PIPELINE EM CÓDIGO VETORIZAÇÃO + CLASSIFICADOR EM UM OBJETO

*Bloco 07*

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
# Tudo em um único fit/predict
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
# Salvar o pipeline inteiro
import joblib
joblib.dump(pipe, 'sentiment_model.joblib')
# Carregar e usar em produção
pipe = joblib.load('sentiment_model.joblib')
y_pred = pipe.predict(['Adorei o produto!', 'Péssimo atendimento.'])
# array([1, 0])

## Slide 33 — GRIDSEARCHCV NO PIPELINE TUNING DE VETORIZAÇÃO + CLASSIFICADOR JUNTOS

*Bloco 08*

In [ ]:
from sklearn.model_selection import GridSearchCV
# Tune AMBOS os componentes do pipeline
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [3, 5, 10],
    'tfidf__max_df': [0.85, 0.95],
    'clf__C': [0.1, 1.0, 10.0],
    'clf__penalty': ['l1', 'l2'],
}
grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,                  # 5-fold cross-validation
    scoring='f1_macro',    # otimiza F1 macro
    n_jobs=-1,             # usa todos os CPUs
)
grid.fit(X_train, y_train)
print(f'Melhores parâmetros: {grid.best_params_}')
print(f'Melhor F1 macro CV: {grid.best_score_:.3f}')
print(f'Test F1 macro: {grid.score(X_test, y_test):.3f}')

## Slide 34 — VALIDAÇÃO CRUZADA ESTRATIFICADA PRESERVANDO PROPORÇÕES DE CLASSE

*Bloco 09*

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    pipe, X_train, y_train,
    cv=skf,
    scoring='f1_macro',
    n_jobs=-1
)
print(f'F1 macro por fold: {scores}')
print(f'Média: {scores.mean():.3f} ± {scores.std():.3f}')
# Em GridSearchCV, basta passar StratifiedKFold no parâmetro cv
# (o default cv=5 já faz isso para classificação)

## Slide 40 — CLASSIFICATION_REPORT MÉTRICAS COMPLETAS EM UMA CHAMADA

*Bloco 10*

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
y_pred = pipe.predict(X_test)
# Report textual completo
print(classification_report(y_test, y_pred,
      target_names=['negativo', 'positivo']))
# Matriz de confusão visual
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu',
            xticklabels=['neg', 'pos'],
            yticklabels=['neg', 'pos'])
plt.xlabel('Previsto'); plt.ylabel('Real')
plt.title('Matriz de Confusão')
plt.show()

## Slide 49 — BIO TAGGING O ESQUEMA PADRÃO PARA ANOTAÇÃO DE NER

*Bloco 11*

In [ ]:
# Exemplo: "Maria Silva trabalha em São Paulo"
# Tokens:    [Maria]   [Silva]   [trabalha]  [em]  [São]    [Paulo]
# Tags BIO:  [B-PER]   [I-PER]   [O]         [O]   [B-LOC]  [I-LOC]
# Como ler:
# - 'Maria' começa entidade do tipo PER (B-PER)
# - 'Silva' continua a mesma entidade PER (I-PER)
# - 'trabalha', 'em' não são entidades (O)
# - 'São' começa entidade do tipo LOC (B-LOC)
# - 'Paulo' continua a mesma entidade LOC (I-LOC)
# Variantes: BIOES (adiciona E=End, S=Single)
# BIO é o padrão por simplicidade

## Slide 50 — NER COM SPACY EM PT-BR PT_CORE_NEWS_LG PRÉ-TREINADO

*Bloco 12*

In [ ]:
import spacy
nlp = spacy.load('pt_core_news_lg')
texto = '''
Ontem comprei um iPhone 15 na Magazine Luiza por R$ 5.999.
A entrega foi feita em São Paulo no dia 20 de outubro pela Fedex.
Atendimento da Apple Brasil foi excelente.
'''
doc = nlp(texto)
print('Entidades encontradas:')
for ent in doc.ents:
    print(f'  {ent.text:<25} | {ent.label_:<6} | '
          f'start={ent.start_char}, end={ent.end_char}')
# Acesso programático: doc.ents é tupla de Span
# Cada Span tem .text, .label_, .start_char, .end_char,
# .start (token), .end (token), .root, etc.

## Slide 51 — SAÍDA DO NER EM PT-BR ENTIDADES RECONHECIDAS PELO SPACY

*Bloco 13*

In [ ]:
# Saída ilustrativa:
# Entidades encontradas:
#   iPhone 15                 | MISC   | start=21,  end=30
#   Magazine Luiza            | ORG    | start=33,  end=47
#   R$ 5.999                  | MONEY  | start=51,  end=59
#   São Paulo                 | LOC    | start=86,  end=95
#   20 de outubro             | DATE   | start=104, end=117
#   Fedex                     | ORG    | start=124, end=129
#   Apple Brasil              | ORG    | start=148, end=160
# Em uma única chamada:
# - 7 entidades extraídas
# - 4 tipos diferentes (MISC, ORG, MONEY, LOC, DATE)
# - Posições de char para destaque visual
# - Pronto para alimentar dashboard ou banco de dados
# Visualizar em Jupyter:
from spacy import displacy
displacy.render(doc, style='ent')

## Slide 52 — AVALIAÇÃO DE NER POR ENTIDADE, NÃO POR TOKEN

*Bloco 14*

In [ ]:
# pip install seqeval
from seqeval.metrics import classification_report as ner_report
# y_true e y_pred são listas de listas de tags BIO
# Exemplo:
# y_true[0] = ['O', 'O', 'B-PER', 'I-PER', 'O', 'B-LOC']
# y_pred[0] = ['O', 'O', 'B-PER', 'I-PER', 'O', 'B-LOC']
# [notebook] exemplo mínimo de tags BIO para rodar a célula
y_true = [['O', 'O', 'B-PER', 'I-PER', 'O', 'B-LOC'], ['B-ORG', 'O', 'O', 'B-LOC']]
y_pred = [['O', 'O', 'B-PER', 'I-PER', 'O', 'B-LOC'], ['B-ORG', 'O', 'O', 'O']]
print(ner_report(y_true, y_pred))
# Saída ilustrativa:
#               precision    recall  f1-score   support
#          LOC       0.92      0.89      0.90       150
#          ORG       0.85      0.81      0.83       200
#          PER       0.94      0.93      0.93        80
#    micro avg       0.89      0.86      0.87       430
#    macro avg       0.90      0.88      0.89       430

## Slide 56 — PIPELINE COMPLETO TF-IDF + LOGREG + NER EM PARALELO

*Bloco 15*

In [ ]:
# [notebook] split de treino para o GridSearch
reviews_train, labels_train = X_train, y_train
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import spacy
# 1) Modelo de classificação de sentimento
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
# 2) GridSearch para tuning
param_grid = {
    'tfidf__min_df': [3, 5, 10],
    'tfidf__max_df': [0.85, 0.95],
    'clf__C': [0.1, 1.0, 10.0],
}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
grid.fit(reviews_train, labels_train)
print(f'Melhores parâmetros: {grid.best_params_}')
print(f'Melhor F1 macro: {grid.best_score_:.3f}')

## Slide 57 — FUNÇÃO ENRIQUECER_REVIEW SENTIMENTO + ENTIDADES EM UMA CHAMADA

*Bloco 16*

In [ ]:
# 3) NER em paralelo no mesmo texto
nlp = spacy.load('pt_core_news_lg')
def enriquecer_review(texto):
    # Sentiment via pipeline treinado
    sentimento = grid.predict([texto])[0]
    # Entidades via spaCy
    doc = nlp(texto)
    entidades = [(ent.text, ent.label_) for ent in doc.ents]
    return {
        'texto': texto,
        'sentimento': int(sentimento),
        'entidades': entidades
    }
# Uso
resultado = enriquecer_review(
    'Adorei o iPhone 15 da Magazine Luiza, entrega rápida em SP!'
)
print(resultado)
# {'texto': '...', 'sentimento': 1,
#  'entidades': [('iPhone 15', 'MISC'),
#                ('Magazine Luiza', 'ORG'), ('SP', 'LOC')]}

---
# Aula 06

In [ ]:
# [notebook] a Aula 02 reutilizou o nome `reviews` com listas pequenas; restauramos o corpus de exemplo
reviews = reviews_corpus

## Slide 19 — LDA EM GENSIM — CÓDIGO COMPLETO REVIEWS EM PT-BR, DO BOW AO COHERENCE

*Bloco 01*

In [ ]:
from gensim import corpora
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel
# 'tokens_limpos' = lista de listas de tokens (output Aula 2)
# Ex.: [["produto", "bom"], ["atendimento", "ruim"], ...]
# 1) Criar dicionário e corpus em BoW
dicionario = corpora.Dictionary(tokens_limpos)
dicionario.filter_extremes(no_below=10, no_above=0.5)
corpus_bow = [dicionario.doc2bow(d) for d in tokens_limpos]
# 2) Treinar LDA com 10 tópicos
lda = LdaModel(corpus=corpus_bow, id2word=dicionario, num_topics=10,
               passes=20, alpha="auto", eta="auto", random_state=42)
# 3) Inspecionar tópicos descobertos
for idx, topic in lda.print_topics(num_words=8):
    print(f"Tópico {idx}: {topic}")
# 4) Coerência (qualidade dos tópicos)
coh = CoherenceModel(model=lda, texts=tokens_limpos,
                     dictionary=dicionario, coherence="c_v", processes=1)
print(f"Coherence (c_v): {coh.get_coherence():.3f}")

## Slide 38 — PYLDAVIS — O PADRÃO DE VISUALIZAÇÃO A FERRAMENTA QUE VENDEU LDA A CIENTISTAS DE DADOS

*Bloco 02*

In [ ]:
# pip install pyLDAvis
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
# Preparar visualização (lda, corpus_bow, dicionario do slide anterior)
vis = gensimvis.prepare(lda, corpus_bow, dicionario, mds="mmds")  # [notebook] evita erro de número complexo no PCoA padrão
pyLDAvis.enable_notebook()
pyLDAvis.display(vis)
# Visualização interativa exibida direto no notebook.

> ℹ️ **Adicionado no notebook:** as células de BERTopic baixam o modelo `paraphrase-multilingual-MiniLM-L12-v2` (~470 MB) na primeira execução e levam alguns minutos. Com as reviews sintéticas os tópicos serão pouco interessantes; troque `reviews` pelo seu corpus real.

## Slide 53 — BERTOPIC EM PT-BR — CÓDIGO COMPLETO SETUP COM SENTENCE TRANSFORMER MULTILINGUAL

*Bloco 03*

In [ ]:
# pip install bertopic sentence-transformers
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
# Modelo de embeddings em português (multilingual)
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
# BERTopic com configuração padrão
topic_model = BERTopic(
    embedding_model=embedder,
    language="multilingual",
    min_topic_size=20,        # ignora tópicos com menos de 20 docs
    nr_topics="auto",
)
# 'reviews' é uma lista de strings em PT
topics, probs = topic_model.fit_transform(reviews)
# Inspecionar tópicos
print(topic_model.get_topic_info().head(10))
# Topic  Count                          Name
#    -1   1245   -1_geral_outros_diversos
#     0    832     0_entrega_prazo_atraso
#     1    721     1_qualidade_produto_bom
#     2    615     2_atendimento_suporte_problema

## Slide 54 — BERTOPIC — VISUALIZAÇÕES NATIVAS INSPIRADAS NO PYLDAVIS E MAIS RICAS AINDA

*Bloco 04*

In [ ]:
# Mapa de tópicos em 2D (similar ao pyLDAvis)
fig = topic_model.visualize_topics()
fig.show()
# Barras horizontais com palavras-chave por tópico
fig = topic_model.visualize_barchart(top_n_topics=10)
fig.show()
# Hierarquia de tópicos (dendrograma)
fig = topic_model.visualize_hierarchy(top_n_topics=20)
fig.show()
# Heatmap de similaridade entre tópicos
fig = topic_model.visualize_heatmap(top_n_topics=15)
fig.show()
# Essas visualizações são especialmente úteis em apresentações
# para times de produto, marketing ou C-level que precisam
# entender o que está sendo discutido nos dados.

## Slide 58 — BERTOPIC VS LDA — COMPARAÇÃO DIRETA CÓDIGO DE BENCHMARK EM CORPUS DE REVIEWS PT-BR

*Bloco 05*

In [ ]:
import time
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel
# 1) LDA via Gensim
t0 = time.time()
lda = LdaModel(corpus=corpus_bow, id2word=dicionario,
               num_topics=10, passes=20, random_state=42)
tempo_lda = time.time() - t0
coh_lda = CoherenceModel(model=lda, texts=tokens_limpos,
                         dictionary=dicionario, coherence="c_v", processes=1).get_coherence()
# 2) BERTopic
t0 = time.time()
topic_model = BERTopic(embedding_model=embedder,
                       language="multilingual", min_topic_size=20)
topics, _ = topic_model.fit_transform(reviews)
tempo_bertopic = time.time() - t0
# Coerência manual: top words de cada tópico
top_words_bert = [[w for w, _ in topic_model.get_topic(t)[:10]]
                  for t in topic_model.get_topics() if t != -1]
coh_bert = CoherenceModel(topics=top_words_bert, texts=tokens_limpos,
                          dictionary=dicionario, coherence="c_v", processes=1).get_coherence()
print(f"{'Modelo':<12} {'Tempo (s)':<12} {'Coherence':<12}")
print(f"{'LDA':<12} {tempo_lda:<12.1f} {coh_lda:<12.3f}")
print(f"{'BERTopic':<12} {tempo_bertopic:<12.1f} {coh_bert:<12.3f}")
# Saída em corpus de 10k reviews:
# LDA          45.2          0.428
# BERTopic     312.8         0.567   ← mais coerente, mais lento